# 08. Две ямы: распределение Больцмана и свободная энергия

**Связь с лекциями:** потенциальная энергия и равновесные вероятности. **Время:** 25–30 минут.

Независимые копии частицы в потенциале $U(x)=A[(x/b)^2-1]^2+h x/b$. Это параллельные реализации одной модели, не взаимодействующая жидкость. Слабое удержание по y,z не связано с x. Из гистограммы x восстанавливаем свободную энергию; в данной разделимой модели она совпадает с U(x) с точностью до константы. В многомерной молекуле такое совпадение обычно отсутствует.

**Запуск:** Colab → «Среда выполнения → Выполнить все». Выбирать GPU не нужно. Все исходные системы создаются кодом; загрузка PDB и ключи доступа не требуются. Выполняйте ячейки сверху вниз. После изменения параметров повторите зависимые ячейки.

**Обозначения:** энергии в kJ/mol, длины в nm, время в ps; 1 fs = 0.001 ps.

In [ ]:
# Установка фиксированной версии: CPU достаточно, GPU не требуется.
%pip -q install openmm==8.3.1 numpy matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openmm as mm
from openmm import unit
from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation
import platform
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True, 'font.size': 11})
rng = np.random.default_rng(2026)
# Маленькие учебные системы: Reference исключает накладные расходы GPU.
backend = mm.Platform.getPlatformByName('Reference')
R = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole/unit.kelvin)
print('OpenMM', mm.__version__, '| Python', platform.python_version(), '|', backend.getName())
# Числовые единицы ниже: nm, ps, dalton, kJ/mol, K.
def vv(dt):
    it = mm.CustomIntegrator(dt*unit.picoseconds)
    it.addComputePerDof('v', 'v+0.5*dt*f/m')
    it.addComputePerDof('x', 'x+dt*v')
    it.addComputePerDof('v', 'v+0.5*dt*f/m')
    return it

def state(ctx):
    st = ctx.getState(getPositions=True, getVelocities=True, getEnergy=True)
    x = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    v = st.getVelocities(asNumpy=True).value_in_unit(unit.nanometer/unit.picosecond)
    u = st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    return x, v, u

def movie(frames, title='', bonds=(), interval=60):
    """Небольшая XY-проекция. Показанные траектории могут быть трёхмерными."""
    a = np.asarray(frames)[::max(1, len(frames)//100)]
    fig, ax = plt.subplots(figsize=(5, 4))
    lo=a[:,:,:2].min(axis=(0,1)); hi=a[:,:,:2].max(axis=(0,1))
    pad=max(float(np.max(hi-lo))*0.15,0.1)
    ax.set(xlim=(lo[0]-pad,hi[0]+pad),ylim=(lo[1]-pad,hi[1]+pad),
           xlabel='x, nm',ylabel='y, nm',title=title,aspect='equal')
    dots=ax.scatter(a[0,:,0],a[0,:,1],s=45,c=np.arange(a.shape[1]),cmap='viridis')
    lines=[ax.plot([],[],color='gray')[0] for _ in bonds]
    def update(i):
        dots.set_offsets(a[i,:,:2])
        for line,(j,k) in zip(lines,bonds): line.set_data(a[i,[j,k],0],a[i,[j,k],1])
        return [dots]+lines
    ani=FuncAnimation(fig,update,frames=len(a),interval=interval,blit=True)
    html=ani.to_jshtml(); plt.close(fig); display(HTML(html))


## Модель и теоретическое распределение

Сначала предскажите, какая яма заселена сильнее при h > 0. Высота барьера и температура подобраны так, чтобы переходы были видны за короткий расчёт.

Попробуйте повторить расчет при h = 10

In [ ]:
N,A,b,h,T=128,5.,.2,1.,300.

#N,A,b,h,T=128,5.,.2,0,300.
#N,A,b,h,T=128,20.,.2,1.,300.

xgrid=np.linspace(-.36,.36,401)
def potential(x):return A*((x/b)**2-1)**2+h*x/b
U=potential(xgrid);prob=np.exp(-(U-U.min())/(R*T));prob/=np.trapz(prob,xgrid)
fig,ax=plt.subplots(1,2,figsize=(12,4));ax[0].plot(xgrid,U);ax[0].set(xlabel='x, nm',ylabel='U, kJ/mol')
ax[1].plot(xgrid,prob);ax[1].set(xlabel='x, nm',ylabel='p(x), nm⁻¹');plt.show()


## Выборка Ланжевена

Начинаем все копии в левой яме. Первые 10 ps отбрасываем. Сравниваем две половины производственной части: это диагностика, а не доказательство полной сходимости.

In [ ]:
sys=mm.System()
for _ in range(N):sys.addParticle(12.)
f=mm.CustomExternalForce('A*((x/b)^2-1)^2+h*x/b+0.5*100*(y*y+z*z)')
for key,val in [('A',A),('b',b),('h',h)]:f.addGlobalParameter(key,val)
for i in range(N):f.addParticle(i,[])
sys.addForce(f);it=mm.LangevinMiddleIntegrator(T,5.,.002);it.setRandomNumberSeed(2026)
ctx=mm.Context(sys,it,backend);pos=np.zeros((N,3));pos[:,0]=-b
ctx.setPositions(pos);ctx.setVelocitiesToTemperature(T*unit.kelvin,2026)
it.step(5000)
samples=[];frames=[]
for j in range(1500):
    it.step(20);x,v,u=state(ctx);samples.append(x[:,0]);frames.append(x[:12])
samples=np.array(samples);del ctx,it
print('Доля x>0, первая и вторая половина:',(samples[:750]>0).mean(),(samples[750:]>0).mean())
movie(frames,'12 независимых копий частицы: XY-проекция')


## Гистограмма, свободная энергия и населённости

Не добавляем искусственные вероятности в пустые бины: там F не определена по данным. Сравниваем населённости интегрированием распределения, а не только по глубине двух минимумов.

In [ ]:
density,edges=np.histogram(samples,bins=70,range=(-.36,.36),density=True)
centers=(edges[1:]+edges[:-1])/2;mask=density>0
F=np.full_like(density,np.nan);F[mask]=-R*T*np.log(density[mask]);F-=np.nanmin(F)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(xgrid,prob,label='Теория');ax[0].stairs(density,edges,label='MD');ax[0].legend();ax[0].set(xlabel='x, nm',ylabel='p(x)')
ax[1].plot(centers,F,'o',ms=3,label='−RT ln p');ax[1].plot(xgrid,U-U.min(),label='U−Umin');ax[1].set(xlabel='x, nm',ylabel='kJ/mol',ylim=(0,15));ax[1].legend();plt.show()
right=xgrid>=0;pr=np.trapz(prob[right],xgrid[right]);observed=(samples>0).mean()
print('P(x>0), теория:',round(pr,3),'MD:',round(observed,3))
assert abs(observed-pr)<.10, 'Слишком большое расхождение: увеличьте выборку и проверьте параметры.'
print('Проверка населённости в пределах 0.10 пройдена. Это учебный допуск, не доверительный интервал.')


## Этропийный вклад поперечной координаты y

In [ ]:
# Энтропийный вклад поперечной координаты y.
# Используются импорты и A, b, T, R из предыдущих ячеек блокнота.

h_demo = 0.0                 # Симметричный энергетический профиль вдоль x
k0, kz = 100.0, 100.0        # kJ/mol/nm²
s = 2.0                     # Сила зависимости ky от x; s=0 — исходный случай
N_demo = 128
dt_demo = 0.001              # ps
rng_demo = np.random.default_rng(2026)


def U0_demo(x):
    return A * ((x / b)**2 - 1)**2 + h_demo * x / b


def ky_demo(x):
    # Положительная и ограниченная жёсткость.
    return k0 * np.exp(s * np.tanh(x / b))


def F_demo(x):
    # Аналитическая свободная энергия после интегрирования по y и z.
    # Определена с точностью до аддитивной константы.
    return U0_demo(x) + 0.5 * R * T * np.log(ky_demo(x) / k0)


# --- 1. Система OpenMM ---

system_demo = mm.System()
for _ in range(N_demo):
    system_demo.addParticle(12.0)

force_demo = mm.CustomExternalForce(
    "A*((x/b)^2-1)^2 + h*x/b"
    " + 0.5*k0*exp(s*tanh(x/b))*y^2"
    " + 0.5*kz*z^2"
)

for name, value in [
    ("A", A), ("b", b), ("h", h_demo),
    ("k0", k0), ("kz", kz), ("s", s)
]:
    force_demo.addGlobalParameter(name, value)

for i in range(N_demo):
    force_demo.addParticle(i, [])

system_demo.addForce(force_demo)

it_demo = mm.LangevinMiddleIntegrator(T, 5.0, dt_demo)
it_demo.setRandomNumberSeed(2026)

ctx_demo = mm.Context(
    system_demo, it_demo,
    mm.Platform.getPlatformByName("Reference")
)

# Половина независимых копий начинает в каждой яме.
# Начальные y и z выбираем из условных равновесных распределений.
p0 = np.zeros((N_demo, 3))
p0[:, 0] = np.where(np.arange(N_demo) < N_demo // 2, -b, b)
p0[:, 1] = (
    rng_demo.normal(size=N_demo)
    * np.sqrt(R * T / ky_demo(p0[:, 0]))
)
p0[:, 2] = rng_demo.normal(size=N_demo) * np.sqrt(R * T / kz)

ctx_demo.setPositions(p0)
ctx_demo.setVelocitiesToTemperature(T * unit.kelvin, 2026)

# Прогрев: 20 ps.
it_demo.step(20000)

# Производственная часть: 60 ps; сохраняем координаты каждые 0.02 ps.
xy_samples = []
for _ in range(3000):
    it_demo.step(20)
    xyz = (
        ctx_demo.getState(getPositions=True)
        .getPositions(asNumpy=True)
        .value_in_unit(unit.nanometer)
    )
    xy_samples.append(xyz[:, :2].copy())

xy_samples = np.asarray(xy_samples)
del ctx_demo, it_demo

x_samples = xy_samples[:, :, 0]


# --- 2. Теоретическое распределение и гистограмма MD ---

xg = np.linspace(-0.45, 0.45, 901)
ug = U0_demo(xg)
fg = F_demo(xg)

weights = np.exp(-(fg - fg.min()) / (R * T))
pg = weights / (weights.sum() * (xg[1] - xg[0]))

counts, edges = np.histogram(
    x_samples, bins=75, range=(xg[0], xg[-1])
)
xc = 0.5 * (edges[:-1] + edges[1:])
density = counts / (counts.sum() * np.diff(edges))

# Не восстанавливаем F в почти непосещённых бинах.
valid = counts >= 30
Fhist = np.full_like(density, np.nan)
Fhist[valid] = -R * T * np.log(density[valid])

# Свободная энергия определена с точностью до константы.
# Выравниваем MD и теорию одной константой по посещённым бинам.
Fcurve = fg - fg.min()
target = F_demo(xc) - fg.min()
offset = np.average(
    Fhist[valid] - target[valid],
    weights=counts[valid]
)
Fhist -= offset


# --- 3. Визуализация ---

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# Карта U(x,y,0): слева поперечная яма шире.
xx, yy = np.meshgrid(
    np.linspace(-0.36, 0.36, 160),
    np.linspace(-0.65, 0.65, 180)
)
landscape = U0_demo(xx) + 0.5 * ky_demo(xx) * yy**2

im = ax[0].contourf(
    xx, yy, landscape,
    levels=np.linspace(-1, 15, 33),
    extend="both", cmap="viridis"
)
points = xy_samples[::100].reshape(-1, 2)
ax[0].scatter(
    points[:, 0], points[:, 1],
    s=1, c="white", alpha=0.2
)
ax[0].set(
    xlabel="x, nm", ylabel="y, nm",
    title="U(x,y,0): справа канал уже",
    xlim=(-0.36, 0.36), ylim=(-0.65, 0.65)
)
fig.colorbar(im, ax=ax[0], label="kJ/mol")

# Распределение координаты x.
ax[1].stairs(density, edges, label="MD")
ax[1].plot(xg, pg, "k--", label="Теория с энтропией")
ax[1].set(xlabel="x, nm", ylabel="P(x), nm⁻¹")
ax[1].legend()

# Потенциальная энергия и свободная энергия.
ax[2].plot(xg, ug - ug.min(), label="U(x,0,0)")
ax[2].plot(
    xg, Fcurve, "k--",
    label="U₀ + (RT/2) ln(ky/k₀)"
)
ax[2].plot(
    xc, Fhist, "o", ms=3,
    label="−RT ln P(x), MD"
)
ax[2].set(
    xlabel="x, nm", ylabel="kJ/mol",
    ylim=(-0.5, 12)
)
ax[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Населённости и простая диагностика выборки.
predicted = pg[xg > 0].sum() / pg.sum()
observed = (x_samples > 0).mean()

print(f"P(x>0): теория = {predicted:.3f}; MD = {observed:.3f}")
half = len(x_samples) // 2
print(
    "MD по двум половинам:",
    round((x_samples[:half] > 0).mean(), 3),
    round((x_samples[half:] > 0).mean(), 3)
)

### Почему профиль свободной энергии отличается от сечения потенциала?

Рассмотрим частицу в потенциале

$$
U(x,y,z)=U_0(x)+\frac{1}{2}k_y(x)y^2+\frac{1}{2}k_z z^2,
$$

где потенциал вдоль координаты $x$ имеет форму двойной ямы:

$$
U_0(x)=A\left[\left(\frac{x}{b}\right)^2-1\right]^2
+h\frac{x}{b}.
$$

В примере коэффициент удержания по $y$ зависит от положения частицы:

$$
k_y(x)=k_0\exp\left[s\tanh\left(\frac{x}{b}\right)\right],
$$

а $k_z$ остаётся постоянным. Все энергии записаны на моль, поэтому в распределении Больцмана используется $RT$.

### От распределения Больцмана к распределению координаты $x$

В каноническом ансамбле равновесная плотность вероятности координат равна

$$
p(x,y,z)=\frac{1}{Z}
\exp\left[-\frac{U(x,y,z)}{RT}\right],
$$

где $Z$ — конфигурационный нормировочный интеграл.

Гистограмма координаты $x$ учитывает все значения $y$ и $z$. Поэтому соответствующая плотность вероятности получается интегрированием:

$$
P(x)=\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}
p(x,y,z)\,dy\,dz.
$$

Подставляя потенциал, получаем

$$
P(x)=\frac{1}{Z}
\exp\left[-\frac{U_0(x)}{RT}\right]
\int_{-\infty}^{\infty}
\exp\left[-\frac{k_y(x)y^2}{2RT}\right]dy
\int_{-\infty}^{\infty}
\exp\left[-\frac{k_z z^2}{2RT}\right]dz.
$$

Используем гауссов интеграл:

$$
\int_{-\infty}^{\infty}
\exp\left[-\frac{kq^2}{2RT}\right]dq
=
\sqrt{\frac{2\pi RT}{k}},
\qquad k>0.
$$

Следовательно,

$$
P(x)=\frac{1}{Z}
\exp\left[-\frac{U_0(x)}{RT}\right]
\sqrt{\frac{2\pi RT}{k_y(x)}}
\sqrt{\frac{2\pi RT}{k_z}}.
$$

Поскольку $k_z$ постоянно, все множители, не зависящие от $x$, можно включить в нормировку:

$$
\boxed{
P(x)\propto
\frac{\exp[-U_0(x)/(RT)]}{\sqrt{k_y(x)}}.
}
$$

### Профиль свободной энергии

Потенциал средней силы вдоль $x$ определяется как

$$
F(x)=-RT\ln\left[\frac{P(x)}{P_{\mathrm{ref}}}\right],
$$

где $P_{\mathrm{ref}}$ — произвольная постоянная с размерностью плотности вероятности. Она делает аргумент логарифма безразмерным и задаёт только нулевой уровень энергии.

Подставляя найденное распределение, получаем

$$
\boxed{
F(x)=U_0(x)
+\frac{RT}{2}\ln\left[\frac{k_y(x)}{k_0}\right]+C.
}
$$

Здесь $C$ не зависит от $x$. Для выбранной зависимости $k_y(x)$:

$$
\boxed{
F(x)=U_0(x)
+\frac{RTs}{2}\tanh\left(\frac{x}{b}\right)+C.
}
$$

Таким образом, сечение потенциала

$$
U(x,0,0)=U_0(x)
$$

и профиль свободной энергии отличаются зависящим от $x$ вкладом, связанным с поперечными флуктуациями.

### Физический смысл: вклад энтропии

При фиксированном $x$ распределение координаты $y$ является гауссовым:

$$
\left\langle y^2\right\rangle_x=\frac{RT}{k_y(x)},
\qquad
\sigma_y(x)=\sqrt{\frac{RT}{k_y(x)}}.
$$

Чем меньше $k_y(x)$, тем шире область доступных значений $y$. У частицы появляется больше доступных конфигураций, а энтропия увеличивается.

Разность поперечных энтропий относительно точки $x_*$ равна

$$
S_\perp(x)-S_\perp(x_*)
=
-\frac{R}{2}
\ln\left[\frac{k_y(x)}{k_y(x_*)}\right].
$$

При этом средняя энергия каждой поперечной гармонической степени свободы остаётся равной $RT/2$:

$$
\left\langle \frac{1}{2}k_y(x)y^2\right\rangle_x
=
\frac{RT}{2}.
$$

Поэтому различие поперечного вклада в свободную энергию имеет энтропийное происхождение: более жёсткое удержание уменьшает энтропию и повышает свободную энергию.

### Что покажет пример?

При $h=0$ сечение $U(x,0,0)$ симметрично и имеет одинаковую глубину ям. Однако при $s>0$ удержание по $y$ сильнее справа:

$$
k_y(b)>k_y(-b).
$$

Поэтому правая область имеет меньшую энтропию и оказывается менее населённой.

В точках $x=\pm b$ разность свободных энергий равна

$$
F(b)-F(-b)=RTs\tanh(1),
$$

а отношение плотностей вероятности составляет

$$
\frac{P(b)}{P(-b)}
=
\exp[-s\tanh(1)].
$$

Это отношение плотностей в двух точках. Вероятности находиться в правой или левой области определяются интегралами:

$$
P_{\mathrm{right}}=\int_0^\infty P(x)\,dx,
\qquad
P_{\mathrm{left}}=\int_{-\infty}^0 P(x)\,dx.
$$

При $s=0$ коэффициент $k_y$ становится постоянным, энтропийная поправка перестаёт зависеть от $x$, и профили $F(x)$ и $U(x,0,0)$ вновь совпадают с точностью до аддитивной константы.

## Самостоятельная работа

1. Установите h=0: сравните населённости ям.
2. Увеличьте A в четыре раза и начните все частицы слева. Почему гистограмма может не совпасть с теорией?

**Ориентиры:** симметрия даёт равные равновесные населённости; высокий барьер мешает выборке; интегрирование поперечной координаты создаёт энтропийный вклад.

## Источники

[OpenMM: интеграторы](https://docs.openmm.org/latest/userguide/theory/04_integrators.html). [OpenMM: стандартные силы](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html); [Custom Forces](https://docs.openmm.org/latest/userguide/theory/03_custom_forces.html).